# DATA3010 · Exam 1
### Friday, October 9, 2026 · weeks 1–4 · 100 points

**Name:** *Luna DesRoches*

**Work time:** 3:05–4:35 PM. Upload to Brightspace by **4:40 PM**.

**Rules**
- Allowed: your laptop, your own course notebooks and notes, Brightspace course materials, the pandas and scikit-learn documentation.
- Not allowed: any AI tool (ChatGPT, Claude, Gemini, Copilot, or AI built into your editor or Colab), messaging, or help from anyone.
- Write all code yourself. Use the variable names given in each task.
- Partial credit is given for a correct approach, so if something fails, leave your code in and write one line saying what you were trying to do.

**Before you start:** run the two cells below. The first must print `Setup complete.`, the second must print `Rows, columns: (652, 10)`.

**To submit:** Kernel → Restart and Run All, check there are no errors, save, and upload this `.ipynb` file to Brightspace.

In [1]:
import numpy as np
import pandas as pd
from IPython.display import display

from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, confusion_matrix, precision_score, recall_score
from sklearn.model_selection import cross_val_score, train_test_split
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.tree import DecisionTreeClassifier

pd.set_option("display.max_columns", 20)
pd.set_option("display.width", 120)

RANDOM_STATE = 42
COST_MISSED = 15   # dollars lost on a return the store did not see coming
COST_CHECK = 2     # dollars for a "fit check" message sent before shipping

print("Setup complete.")

Setup complete.


## The data
An online store wants to predict which orders will be **returned**, so it can send a short "fit check" message before shipping. A return it did not see coming costs about **\$15**. A fit-check message costs about **\$2**.

| Column | Meaning |
|---|---|
| `order_id` | Order number |
| `category` | Product category, as typed by staff |
| `price_text` | Price, as shown on the website |
| `discount_pct` | Discount on the order, in percent |
| `days_to_deliver` | Days from order to delivery |
| `past_returns` | How many orders this customer returned before |
| `used_size_guide` | 1 if the customer opened the size guide |
| `store` | Store name |
| `refund_amount` | Amount refunded on this order |
| `returned` | **The answer:** 1 = returned, 0 = kept |

The cell below creates the data. **Do not change it.**

In [2]:
rng = np.random.default_rng(1009)
n = 640
returned = rng.binomial(1, 0.22, n)
cat_true = rng.choice(["clothing", "shoes", "electronics"], n, p=[0.45, 0.30, 0.25])
price = np.where(cat_true == "electronics", rng.uniform(40, 900, n), rng.uniform(15, 160, n)).round(2)
past = np.where(returned == 1, rng.poisson(2.0, n), rng.poisson(0.5, n))
days = np.where(returned == 1, rng.normal(7.5, 2.5, n), rng.normal(4.5, 2.0, n)).round().clip(1, 20)
disc = np.where(returned == 1, rng.choice([0, 10, 20, 30, 40], n, p=[.15, .15, .2, .25, .25]),
                rng.choice([0, 10, 20, 30, 40], n, p=[.4, .25, .2, .1, .05]))
guide = np.where(cat_true == "electronics", 0,
                 np.where(returned == 1, rng.binomial(1, 0.55, n), rng.binomial(1, 0.15, n)))
variants = {"clothing": ["Clothing", "clothing", "CLOTHING ", "Clothes"],
            "shoes": ["Shoes", "shoes", " SHOES"],
            "electronics": ["Electronics", "electronics", "Electronics "]}
cat_raw = [rng.choice(variants[c]) for c in cat_true]

def as_text(p):
    r = rng.random()
    if r < 0.6: return f"${p:,.2f}"
    if r < 0.85: return f"{p:.2f}"
    return f"USD {p:,.2f}"

price_text = [as_text(p) for p in price]
days = days.astype(float)
days[rng.choice(n, 25, replace=False)] = np.nan
days[rng.choice(np.where(~np.isnan(days))[0], 6, replace=False)] = -1
refund = np.where(returned == 1, (price * rng.uniform(0.8, 1.0, n)).round(2), 0.0)

orders = pd.DataFrame({
    "order_id": [f"S{i:04d}" for i in range(1, n + 1)],
    "category": cat_raw, "price_text": price_text,
    "discount_pct": disc, "days_to_deliver": days,
    "past_returns": past, "used_size_guide": guide,
    "store": "Online", "refund_amount": refund, "returned": returned})
extra = orders.sample(12, random_state=3)
orders = pd.concat([orders, extra], ignore_index=True).sample(frac=1, random_state=4).reset_index(drop=True)

print("Rows, columns:", orders.shape)
display(orders.head())

Rows, columns: (652, 10)


,order_id,category,price_text,discount_pct,days_to_deliver,past_returns,used_size_guide,store,refund_amount,returned
0,S0254,Clothing,$131.62,0,4.0,0,0,Online,0.00,0
1,S0203,Clothes,$18.65,20,3.0,1,0,Online,0.00,0
2,S0428,Shoes,$16.21,20,3.0,0,0,Online,0.00,0
3,S0529,Clothes,$158.37,0,2.0,0,0,Online,0.00,0
4,S0448,electronics,254.67,40,8.0,1,0,Online,214.98,1


# Part A · Preprocessing (30 points)

## A1 · Inspect (5 points)

Show the evidence before you change anything:
- the data types of the columns,
- the number of missing values in each column,
- the number of **exact duplicate rows**,
- the share of orders that were returned.

In [3]:
# A1 · your code here
print("orders info", orders.info())

print("\nmissing values", orders.isna().sum())

print("\nduplicates:", orders.duplicated().sum())

print("\n# of orders returned:")
display(orders["returned"].value_counts(normalize=True))

<class 'pandas.DataFrame'>
RangeIndex: 652 entries, 0 to 651
Data columns (total 10 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   order_id         652 non-null    str    
 1   category         652 non-null    str    
 2   price_text       652 non-null    str    
 3   discount_pct     652 non-null    int64  
 4   days_to_deliver  627 non-null    float64
 5   past_returns     652 non-null    int64  
 6   used_size_guide  652 non-null    int64  
 7   store            652 non-null    str    
 8   refund_amount    652 non-null    float64
 9   returned         652 non-null    int64  
dtypes: float64(2), int64(4), str(4)
memory usage: 51.1 KB
orders info None

missing values order_id            0
category            0
price_text          0
discount_pct        0
days_to_deliver    25
past_returns        0
used_size_guide     0
store               0
refund_amount       0
returned            0
dtype: int64

duplicates: 12

# of orders 

returned
0    0.777607
1    0.222393
Name: proportion, dtype: float64

## A2 · Clean the text columns (10 points)

Create `clean`, a copy of `orders` **without the exact duplicate rows**. Then in `clean`:
1. Create a number column `price` from `price_text`. Watch for `$`, `USD`, commas and spaces.
2. Standardize `category` so it has exactly three values: `clothing`, `shoes`, `electronics`.

Print the number of rows in `clean` and the counts of each category.

In [4]:
clean = None
# A2 · your code here

clean = orders.copy().drop_duplicates().reset_index(drop=True)

cat_map = {
    "shoes": "shoes",
    "electronics": "electronics",
    "clothes": "clothing",
    "clothing": "clothing",
}

clean["price"] = (
    clean["price_text"]
    .str.replace("$", "")
    .str.replace("USD", "")
    .str.replace(",", "")
    .str.replace(" ", "")
    .astype(float)
)

clean['category'] = clean['category'].str.strip().str.lower().map(cat_map)

print("Rows in clean:", len(clean))

print("\nCategories: \n",clean["category"].value_counts(dropna=False))

Rows in clean: 640

Categories: 
 category
clothing       304
shoes          187
electronics    149
Name: count, dtype: int64


## A3 · Impossible and missing delivery days (8 points)

In `clean`:
1. Delivery can never take a negative number of days. Set every negative `days_to_deliver` to missing.
2. Create `days_was_missing`: **1** where `days_to_deliver` is now missing, **0** otherwise.

**Do not fill the missing values yet.** You will fill them in Part B, after the split.

Print how many negative values you found and how many values are missing now.

In [5]:
# A3 · your code here
print("sum of negative values: ",(clean['days_to_deliver'] < 0).sum())

clean.loc[clean['days_to_deliver'] < 0, 'days_to_deliver'] = np.nan

clean['days_was_missing'] = clean['days_to_deliver'].isna().astype(int)


print(clean['days_was_missing'].value_counts())

sum of negative values:  6
days_was_missing
0    609
1     31
Name: count, dtype: int64


## A4 · Choose the features (7 points)

Some columns must not be model inputs. Drop every column that is an ID, a constant, already replaced by a cleaner column, or **only known after the order was returned**.

Then:
1. One-hot encode `category` with `pd.get_dummies(..., columns=["category"], dtype=int)`.
2. Create `X` (every remaining column except `returned`) and `y` (`returned`).

Print the list of feature names. In the answer cell, name each dropped column and give a one-line reason.

In [6]:
X = None
y = None

# A4 · your code here
drop_col = ['returned', 'order_id', 'price_text', 'category', 'store', 'refund_amount']


X = clean.drop(columns=drop_col)

y = clean['returned']


**Your answer:**

*I chose to drop the columns returned, order_id, refund_amount, and price_text. These columns either provide the answer or are known at runtime, or are used for identification or are redundant non-int versions of other columns*

# Part B · Classification (55 points)

## B1 · Split, fill, and the baseline (8 points)

1. Split `X` and `y` into `X_train, X_test, y_train, y_test` with `test_size=0.25`, `random_state=RANDOM_STATE`, `stratify=y`.
2. Fill the missing `days_to_deliver` in **both** `X_train` and `X_test` using the median of the **training** rows only.
3. Print the size of each part, the training median you used, and the **test baseline** (the accuracy of always predicting the most common class on the test rows).

In [7]:
X_train = X_test = y_train = y_test = None

# B1 · your code here

class_counts = y.value_counts()
baseline_accuracy = class_counts.max() / len(y)

print(class_counts)
print("Baseline", round(baseline_accuracy, 3))

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=RANDOM_STATE, stratify=y
)

day_train_medians = X_train["days_to_deliver"].median()
day_test_medians = X_test["days_to_deliver"].median()

X_train["days_to_deliver"] = X_train["days_to_deliver"].fillna(day_train_medians)

X_test["days_to_deliver"] = X_test["days_to_deliver"].fillna(day_test_medians)

print(
    "X_train: \n",
    f"Size: {X_train.size}\n",
    f"Medians: {day_train_medians}\n",
    f"Accuracy: {baseline_accuracy}\n",
)
print(
    "X_test: \n",
    f"Size: {X_test.size}\n",
    f"Medians: {day_test_medians}\n",
    f"Accuracy: {baseline_accuracy}\n",
)

returned
0    497
1    143
Name: count, dtype: int64
Baseline 0.777
X_train: 
 Size: 2880
 Medians: 5.0
 Accuracy: 0.7765625

X_test: 
 Size: 960
 Medians: 5.0
 Accuracy: 0.7765625



## B2 · KNN (10 points)

Build `knn_model`: a pipeline of `StandardScaler()` and `KNeighborsClassifier(n_neighbors=15)`, fitted on the training rows.

Print its **accuracy, precision and recall** on the test rows.

In [8]:
knn_model = None

# B2 · your code here

knn_model = make_pipeline(StandardScaler(), KNeighborsClassifier(n_neighbors=15)).fit(
    X_train, y_train
)

y_pred = knn_model.predict(X_test)
print(f"Accuracy:  {accuracy_score(y_test, y_pred):.3f}")
print(f"Precision: {precision_score(y_test, y_pred):.3f}")
print(f"Recall:    {recall_score(y_test, y_pred):.3f}")

Accuracy:  0.900
Precision: 0.857
Recall:    0.667


## B3 · Choose the tree depth (12 points)

For each depth in `[2, 3, 4, 6, None]`, report:
- training accuracy,
- 5-fold cross-validation accuracy on the **training rows**,
- test accuracy.

Put the results in a DataFrame called `depth_table`, display it, and set `best_depth` to the depth you would choose. Use `random_state=RANDOM_STATE` for every tree. In the answer cell, say why you chose it.

In [9]:
depth_table = None
best_depth = None

# B3 · your code here

records = []
for d in [2, 3, 4, 6, None]:
    clf = DecisionTreeClassifier(max_depth=d, random_state=RANDOM_STATE).fit(
        X_train, y_train
    )
    cv_acc = cross_val_score(clf, X_train, y_train, cv=5).mean()
    records.append(
        {
            "depth": d,
            "train_acc": clf.score(X_train, y_train),
            "cv_acc": cv_acc,
            "test_acc": clf.score(X_test, y_test),
        }
    )

depth_table = pd.DataFrame(records)
display(depth_table)

best_depth = depth_table.loc[depth_table["cv_acc"].idxmax(), "depth"]

best_depth = int(best_depth)

,depth,train_acc,cv_acc,test_acc
0,2.0,0.843750,0.822917,0.85625
1,3.0,0.868750,0.825000,0.88750
2,4.0,0.889583,0.839583,0.86875
3,6.0,0.922917,0.825000,0.86250
4,NaN,1.000000,0.804167,0.80000


<!-- **Your answer:**

*A depth of 4 has the best cv accuracy on the training data.* -->

## B4 · What kind of mistakes? (10 points)

Fit `best_tree`, a `DecisionTreeClassifier(max_depth=best_depth, random_state=RANDOM_STATE)`, on the training rows.

Print its confusion matrix on the test rows and its **precision** and **recall**. In the answer cell, explain in plain words what a false positive and a false negative mean **for this store**, and which one costs more.

In [10]:
best_tree = None

# B4 · your code here

best_tree = DecisionTreeClassifier(max_depth=best_depth, random_state=RANDOM_STATE).fit(
    X_train, y_train

)

y_pred = best_tree.predict(X_test)

print("Confusion Matrix:\n", confusion_matrix(y_test, y_pred))
print(f"Precision: {precision_score(y_test, y_pred):.3f}")
print(f"Recall:    {recall_score(y_test, y_pred):.3f}")

Confusion Matrix:
 [[115   9]
 [ 12  24]]
Precision: 0.727
Recall:    0.667


**Your answer:**

*For this store, false positives could mean predicting an order will be returned when the customer keeps it. While a false negative is when a user returns an item when predicted to keep it. This could affect the stores restocking ability. False negatives cost the store more because they have to refund the user.*

## B5 · Choose a threshold with the costs (15 points)

1. Build `logreg`: a pipeline of `StandardScaler()` and `LogisticRegression(random_state=RANDOM_STATE)`, fitted on the training rows.
2. Get the probability of a return for each test order with `predict_proba`.
3. For each threshold in `[0.2, 0.3, 0.4, 0.5, 0.6]`, flag the orders whose probability is at least the threshold and report: **flagged**, **caught** (true positives), **missed** (false negatives), **precision**, **recall** and **cost**, where

`cost = missed * COST_MISSED + flagged * COST_CHECK`

Put the results in `threshold_table`, display it, and set `best_threshold` to the cheapest one.

In [11]:
logreg = None
threshold_table = None
best_threshold = None

# B5 · your code here

logreg = make_pipeline(
    StandardScaler(), LogisticRegression(random_state=RANDOM_STATE)
).fit(X_train, y_train)

# Didnt have enough time to finish, we should store the probabiliy with logreg.predict_proba(X_test) and then for each threshold, we need to append threshold, flagged, caught, missed, precision, recall, and cost. after that. We will create a dataframe of the rows

# Part C · Decisions (15 points)
Answer in plain language. Use numbers from **your own** results above.

## C1 · Your recommendation (7 points)
Which model, and which threshold if it applies, would you give the store? Use **at least three numbers** from your work. Name one thing that would make you change your mind.

**Your answer:**

*I would recommend the KNN model. It had the best accuracy, recall, and precision of the other models:*

*Accuracy:  0.900, Precision: 0.857, Recall:    0.667*

## C2 · Audit the AI analysis (8 points)
A classmate asks an AI assistant for help and gets this back:

> "I used every column, including `refund_amount`, and reached 97.5% accuracy, which is excellent. I filled the missing delivery days with the median of the whole dataset before splitting. I chose the tree depth with the best test accuracy. For the final model I used the standard threshold of 0.5."

Find **three** problems. For each one: what is wrong, and what should be done instead.

**Your answer:**

1. Refund amount gives the answer away, it is only filled if the user requests a refund
2. When getting the median the delivery days before splitting, it skews the prediction of the model. You should replace the median after the split. 
3. When chosing a model with the best tree depth, you should aim for a higher cv_acc instead of straight accuracy. 

## Before you upload
- Kernel → **Restart and Run All**. Check every cell ran.
- Every answer cell has your writing in it.
- Save, then upload this `.ipynb` file to Brightspace by **4:40 PM**.